# WS01: Platform Basics and Interactions

## Overview
This workshop guides you through the basics of interacting with the FIWARE Orion Context Broker using the `filip` library. You will learn how to:
- Initialize and configure the platform
- Verify platform health status
- Execute CRUD (Create, Read, Update, Delete) operations on entities

## Use Case
We are digitizing an intelligent building with:
- 1 office building
- 3 floors
- 10 office rooms per floor
- Each office room contains: temperature sensor, humidity sensor, CO2 sensor, air ventilation unit, radiator thermostat, and fan coil unit

In this workshop, we start with creating simple room entities to understand the basic CRUD operations.

## Step 1: Setup and Configuration

First, let's import the necessary packages and configure the connection to the Context Broker.

In [1]:
# import package
from filip.clients.ngsi_v2 import ContextBrokerClient
from filip.models.base import FiwareHeaderSecure, DataType
from filip.models.ngsi_v2.context import (
    ContextEntity,
    ContextAttribute,
    NamedContextAttribute,
)
from filip.utils.simple_ql import QueryString
from utils import KeycloakTokenManager

print("Packages imported successfully!")

D:\anaconda3\envs\n5gehworkshop\Lib\site-packages\pydantic\_internal\_fields.py:201: UserWarning: Field name "json" in "HttpCustom" shadows an attribute in parent "Http"
  warnings.warn(
D:\anaconda3\envs\n5gehworkshop\Lib\site-packages\pydantic\_internal\_fields.py:201: UserWarning: Field name "json" in "MqttCustom" shadows an attribute in parent "Mqtt"
  warnings.warn(


Packages imported successfully!


## Step 2: Configure Connection Parameters

Set up the connection details to the Context Broker.

In [2]:
# Configuration parameters
# Host address of Context Broker
CB_URL = "https://n5geh.eonerc.rwth-aachen.de/orion/"

# FIWARE Service and Service Path
SERVICE = "n5geh_demo"


# Create FIWARE header
manager = KeycloakTokenManager(fiware_service=SERVICE)

fiware_header = FiwareHeaderSecure(service=SERVICE,
                                   authorization=f"Bearer {manager.get_access_token()}")

Token expired or missing. Fetching new token...


## Step 3: Create and Initialize the Client

Create the ContextBrokerClient and verify the connection.

In [3]:
# Create the Context Broker client
cb_client = ContextBrokerClient(url=CB_URL, fiware_header=fiware_header)

# Verify platform health status
version_info = cb_client.get_version()
print(version_info)
print("✓ Connection to Context Broker established successfully!")

{'orion': {'version': '3.11.0', 'uptime': '153 d, 4 h, 18 m, 45 s', 'git_hash': 'c505fdc0616ce559c01e19b7130b019d85c4feba', 'compile_time': 'Mon Jan 29 08:45:34 UTC 2024', 'compiled_by': 'root', 'compiled_in': 'buildkitsandbox', 'release_date': 'Mon Jan 29 08:45:34 UTC 2024', 'machine': 'x86_64', 'doc': 'https://fiware-orion.rtfd.io/en/3.11.0/', 'libversions': {'boost': '1_74', 'libcurl': 'libcurl/7.88.1 OpenSSL/3.0.11 zlib/1.2.13 brotli/1.0.9 zstd/1.5.4 libidn2/2.3.3 libpsl/0.21.2 (+libidn2/2.3.3) libssh2/1.10.0 nghttp2/1.52.0 librtmp/2.3 OpenLDAP/2.5.13', 'libmosquitto': '2.0.15', 'libmicrohttpd': '0.9.76', 'openssl': '3.0.11', 'rapidjson': '1.1.0', 'mongoc': '1.24.3', 'bson': '1.24.3'}}}
✓ Connection to Context Broker established successfully!


## Step 4: Create Entities

We'll create two room entities using different approaches. This demonstrates the flexibility of the NGSI-v2 entity model.

### Approach 1: Creating an Entity from a Dictionary

In [ ]:
# Approach 1: Create entity from dictionary
room1_dictionary = {
    "id": "urn:ngsi-ld:Room:001",
    "type": "Room",
    "temperature": {"value": 21.5, "type": "Float"},
    "humidity": {"value": 45, "type": "Integer"},
    "floor": {"value": 1, "type": "Integer"},
}

room1_entity = ContextEntity(**room1_dictionary)

print("Room 1 Entity (created from dictionary):")
print(room1_entity.model_dump_json(indent=2))

### Approach 2: Creating an Entity Using Constructor and Interfaces

In [ ]:
# Approach 2: Create entity using constructor and interfaces
room2_entity = ContextEntity(id="urn:ngsi-ld:Room:002", type="Room")

# Add attributes individually
temp_attr = NamedContextAttribute(name="temperature", value=22.3, type=DataType.FLOAT)
humidity_attr = NamedContextAttribute(name="humidity", value=50, type=DataType.INTEGER)
floor_attr = NamedContextAttribute(name="floor", value=2, type=DataType.INTEGER)

room2_entity.add_attributes([temp_attr, humidity_attr, floor_attr])

print("Room 2 Entity (created using constructor):")
print(room2_entity.model_dump_json(indent=2))

### Create Additional Room Entities

Let's create more room entities for different floors to simulate the building structure.

In [ ]:
# Create rooms for all 3 floors
rooms = []

for floor in range(1, 4):
    for room_number in range(1, 4):  # Create 3 rooms per floor for this example
        room_id = f"urn:ngsi-ld:Room:{floor:02d}{room_number:02d}"
        room = ContextEntity(id=room_id, type="Room")
        
        # Add room attributes
        room.add_attributes([
            NamedContextAttribute(name="temperature", value=20 + floor, type=DataType.FLOAT),
            NamedContextAttribute(name="humidity", value=40 + room_number * 5, type=DataType.INTEGER),
            NamedContextAttribute(name="co2", value=400 + floor * 100, type=DataType.INTEGER),
            NamedContextAttribute(name="floor", value=floor, type=DataType.INTEGER),
            NamedContextAttribute(name="roomNumber", value=room_number, type=DataType.INTEGER),
        ])
        
        rooms.append(room)

print(f"Created {len(rooms)} room entities")
print("\nFirst room example:")
print(rooms[0].model_dump_json(indent=2))

## Step 5: POST Entities to Context Broker (CREATE)

Now let's send these entities to the Context Broker.

In [ ]:
# Check entities before posting
print("Entities in Context Broker before posting:")
existing_entities = cb_client.get_entity_list()
print(f"Count: {len(existing_entities)}")

# Post room 1 and room 2
try:
    cb_client.post_entity(entity=room1_entity)
    print(f"✓ Posted {room1_entity.id}")
except Exception as e:
    print(f"✗ Failed to post {room1_entity.id}: {e}")

try:
    cb_client.post_entity(entity=room2_entity)
    print(f"✓ Posted {room2_entity.id}")
except Exception as e:
    print(f"✗ Failed to post {room2_entity.id}: {e}")

# Post all other rooms
for room in rooms:
    try:
        cb_client.post_entity(entity=room)
        print(f"✓ Posted {room.id}")
    except Exception as e:
        print(f"✗ Failed to post {room.id}: {e}")

## Step 6: READ Entities from Context Broker

Retrieve and display entities using various filtering methods.

### 6.1 Get All Entities

In [ ]:
# Get all entities after posting
all_entities = cb_client.get_entity_list()
print(f"Total entities in Context Broker: {len(all_entities)}")
print("\nAll entity IDs:")
for entity in all_entities:
    print(f"  - {entity.id} (Type: {entity.type})")

### 6.2 Get Entities by ID

In [ ]:
# Get entities by specific ID
room_001 = cb_client.get_entity_list(entity_ids=["urn:ngsi-ld:Room:001"])
print(f"Retrieved entities with ID 'urn:ngsi-ld:Room:001': {len(room_001)} entity")

if room_001:
    print("\nRoom details:")
    print(room_001[0].model_dump_json(indent=2))

### 6.3 Get Entities by Type

In [ ]:
# Get all entities of type "Room"
room_entities = cb_client.get_entity_list(entity_types=["Room"])
print(f"Total 'Room' type entities: {len(room_entities)}")
print("\nRoom IDs:")
for room in room_entities:
    print(f"  - {room.id}")

### 6.4 Get Entities by ID Pattern (Regular Expression)

Use regex patterns to filter entities by ID. For example, get all rooms on floor 1.

In [ ]:
# Get rooms on floor 1 using regex pattern
# Pattern explanation: urn:ngsi-ld:Room:01XX where XX can be any digit
floor1_rooms = cb_client.get_entity_list(id_pattern="^urn:ngsi-ld:Room:01")
print(f"Rooms on floor 1: {len(floor1_rooms)} rooms")
print("\nFloor 1 Room IDs:")
for room in floor1_rooms:
    print(f"  - {room.id}")

### 6.5 Get Entities by Query Expression

Filter entities based on attribute values using SimpleQL query expressions.

In [ ]:
# Get rooms with temperature >= 22
query = QueryString(qs=[("temperature", ">=", 22)])
warm_rooms = cb_client.get_entity_list(q=query)
print(f"Rooms with temperature >= 22: {len(warm_rooms)} rooms")
print("\nWarm rooms:")
for room in warm_rooms:
    temp_attr = room.get_attribute("temperature")
    if temp_attr:
        print(f"  - {room.id}: {temp_attr.value}°C")

### 6.6 Get Attributes of a Specific Entity

In [ ]:
# Get all attributes of a specific entity
attributes = cb_client.get_entity_attributes(entity_id="urn:ngsi-ld:Room:001")
print(f"Attributes of Room:001:")
for attr_name, attr_value in attributes.items():
    print(f"  - {attr_name}: {attr_value}")

### 6.7 Advanced Query: Multiple Conditions

In [ ]:
# Get rooms with multiple conditions
# Rooms with CO2 > 500 AND humidity < 60
query_advanced = QueryString(qs=[
    ("co2", ">", 500),
    ("humidity", "<", 60)
])
filtered_rooms = cb_client.get_entity_list(q=query_advanced)
print(f"Rooms with CO2 > 500 AND humidity < 60: {len(filtered_rooms)} rooms")
print("\nFiltered rooms:")
for room in filtered_rooms:
    co2 = room.get_attribute("co2")
    humidity = room.get_attribute("humidity")
    print(f"  - {room.id}: CO2={co2.value}, Humidity={humidity.value}")

## Step 7: UPDATE Entities (UPDATE)

Modify entity attributes and synchronize changes with the Context Broker.

### 7.1 Update a Single Attribute Value

In [ ]:
# Update temperature of room 1 directly
print(f"Updating temperature of {room1_entity.id}...")
try:
    result = cb_client.update_attribute_value(
        entity_id=room1_entity.id,
        attr_name="temperature",
        value=25.5
    )
    print(f"✓ Temperature updated successfully")
except Exception as e:
    print(f"✗ Failed to update: {e}")

# Verify the update
updated_room = cb_client.get_entity_list(entity_ids=[room1_entity.id])[0]
updated_temp = updated_room.get_attribute("temperature")
print(f"New temperature value: {updated_temp.value}°C")

### 7.2 Update Multiple Attributes at Once

In [ ]:
# Fetch the latest state of room2 from the Context Broker
room2_updated = cb_client.get_entity_list(entity_ids=[room2_entity.id])[0]

# Modify attributes locally
temp_attr = room2_updated.get_attribute("temperature")
temp_attr.value = 23.0
room2_updated.update_attribute([temp_attr])

humidity_attr = room2_updated.get_attribute("humidity")
humidity_attr.value = 55
room2_updated.update_attribute([humidity_attr])

# Send all changes to the Context Broker at once
try:
    cb_client.patch_entity(room2_updated)
    print(f"✓ Multiple attributes of {room2_entity.id} updated successfully")
except Exception as e:
    print(f"✗ Failed to update: {e}")

# Verify the updates
room2_check = cb_client.get_entity_list(entity_ids=[room2_entity.id])[0]
print(f"Updated values:")
print(f"  - Temperature: {room2_check.get_attribute('temperature').value}°C")
print(f"  - Humidity: {room2_check.get_attribute('humidity').value}%")

### 7.3 Add New Attributes to an Entity

In [ ]:
# Fetch the latest state of room1
room1_updated = cb_client.get_entity_list(entity_ids=[room1_entity.id])[0]

# Add new attributes (e.g., occupancy and location)
room1_updated.add_attributes([
    NamedContextAttribute(name="occupancy", value=False, type=DataType.BOOLEAN),
    NamedContextAttribute(name="location", value="Building A", type=DataType.TEXT)
])

# Update the entity in the Context Broker
try:
    cb_client.patch_entity(room1_updated)
    print(f"✓ New attributes added to {room1_entity.id}")
except Exception as e:
    print(f"✗ Failed to add attributes: {e}")

# Verify the new attributes
room1_check = cb_client.get_entity_list(entity_ids=[room1_entity.id])[0]
print(f"Updated attributes of {room1_entity.id}:")
for attr_name, attr_value in room1_check.model_dump()["custom_attributes"].items():
    print(f"  - {attr_name}")

## Step 8: DELETE Operations (DELETE)

Remove entities and attributes from the Context Broker.

### 8.1 Delete a Specific Attribute

In [ ]:
# Delete a specific attribute from an entity
try:
    cb_client.delete_entity_attribute(
        entity_id=room2_entity.id,
        attr_name="floor"
    )
    print(f"✓ Deleted 'floor' attribute from {room2_entity.id}")
except Exception as e:
    print(f"✗ Failed to delete attribute: {e}")

# Verify deletion
room2_check = cb_client.get_entity_list(entity_ids=[room2_entity.id])[0]
remaining_attrs = list(room2_check.model_dump()["custom_attributes"].keys())
print(f"Remaining attributes: {remaining_attrs}")

### 8.2 Delete an Entire Entity

In [ ]:
# Delete entire entities
entities_to_delete = [room1_entity.id, room2_entity.id]

for entity_id in entities_to_delete:
    try:
        cb_client.delete_entity(
            entity_id=entity_id,
            entity_type="Room"
        )
        print(f"✓ Deleted entity: {entity_id}")
    except Exception as e:
        print(f"✗ Failed to delete {entity_id}: {e}")

# Verify deletion
remaining_entities = cb_client.get_entity_list()
print(f"\nRemaining entities in Context Broker: {len(remaining_entities)}")

## Summary

In this workshop, you have learned:

1. **Setup**: Initialized the FIWARE Context Broker connection using the `filip` library
2. **CREATE**: Created entities using two approaches:
   - From dictionaries
   - Using constructor and interfaces
3. **READ**: Retrieved entities using various filtering methods:
   - Get all entities
   - Filter by ID, type, and ID pattern (regex)
   - Query by attribute values (SimpleQL)
   - Get specific attributes
4. **UPDATE**: Modified entities:
   - Updated single attributes
   - Updated multiple attributes
   - Added new attributes
5. **DELETE**: Removed data:
   - Deleted specific attributes
   - Deleted entire entities

These CRUD operations form the foundation for managing IoT entities in FIWARE.

## Exercises

Try these exercises to deepen your understanding:

1. **Create a sensor entity**: Create a temperature sensor entity with attributes like `sensorId`, `location`, `accuracy`, and `lastReading`.

2. **Advanced queries**: 
   - Find all rooms on floor 2
   - Find rooms with humidity between 40% and 60%
   - Find rooms with CO2 levels indicating poor air quality (> 800 ppm)

3. **Batch operations**: Create a script that creates 100 room entities and then performs a bulk query.

4. **Error handling**: Implement try-except blocks for all Context Broker operations to handle potential failures gracefully.